In [ ]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

DATA_PATH  = library_path.parent / 'data'
PLOTS_PATH = library_path.parent / 'plots'
PLOTS_PATH.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

## Latent Class Analysis (LCA)

### Rationale

K-means assumes clusters are spherical and equal-variance — a strong geometric assumption
that may not hold for a mixed dataset of binary, ordinal, and count variables. **LCA**
is a model-based alternative: it assumes each observation belongs to one of $K$ latent
classes, with class membership and item-response probabilities estimated by EM.

LCA is particularly well-suited here because:
- Most features are categorical or ordinal (binary health conditions, employment dummies,
  education, medication count)
- It yields probabilistic class memberships, not hard assignments
- Classes have a direct substantive interpretation (e.g. "young healthy employed",
  "older multimorbid", "working-age with mental health burden")
- Class profiles can be directly compared across datasets to identify which classes
  are over- or under-represented in DAPHNIE vs HSE

### Model selection

Number of classes $K$ selected by BIC (preferred for LCA — penalises model complexity
more strongly than AIC, reducing tendency to over-extract classes). AIC reported for
comparison. Fit evaluated for $K = 2, \ldots, 8$.

### Feature set

The 17-variable predictor set from notebook 04, treated as categorical/ordinal inputs
to the LCA model. Missing values handled by full-information maximum likelihood (FIML)
within the EM algorithm where supported, otherwise complete-case per variable.

### Design

- Fit LCA for $K = 2, \ldots, 8$; select $K$ by BIC
- Extract posterior class probabilities and modal class assignments
- Profile each class: item-response probabilities for all features
- Compare class prevalence across datasets (DAPHNIE vs HSE)
- Cross-tabulate with k-means clusters (notebook 01) and community detection (notebook 03)
  to assess convergent validity

In [ ]:
df = pd.read_csv(DATA_PATH / 'wrangled_data.csv', low_memory=False)
print(f'Full dataset: {len(df):,} rows x {df.shape[1]} columns')
print(df['dataset'].value_counts().to_string())

In [ ]:
# 17-variable predictor set (notebook 04)
FEATURES = [
    'Sex', 'age7cat',
    'eth2cat',
    'emp_cat_Employed', 'emp_cat_Other (Sick/Home/etc)', 'emp_cat_Retired',
    'emp_cat_Student', 'emp_cat_Unemployed',
    'edu_cat_2',
    'paVig', 'paMod',
    'smoke_ecig', 'diabetes',
    'meds_num', 'ill_dis',
    'resp', 'skin',
]
FEATURES = [f for f in FEATURES if f in df.columns]
assert len(FEATURES) == 17, f'Expected 17 features, got {len(FEATURES)}'
print(f'Feature set ({len(FEATURES)}): {FEATURES}')